# Qwen anchor geometry profile sweep

Geometry-only прогон по `short / medium / long` anchor spans.
Смотрим не policy gain, а layerwise crystallization: `first_positive_layer`, `stable_birth_layer`, `max_separation_layer`, group transitions и polarity path.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q --upgrade pip
!pip install -q torch torchvision pillow accelerate einops pytest numpy nbformat "transformers @ git+https://github.com/huggingface/transformers.git@main"


In [ ]:
%cd /content
import os
import subprocess
from pathlib import Path

if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
else:
    %cd /content/ABPT
    !git pull --ff-only

%cd /content/ABPT
print('HEAD =', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import json
from pathlib import Path

MODEL_NAME = 'Qwen/Qwen3.5-4B'
DEVICE = 'cuda'
MAX_LENGTH = 160
PROFILES = ['short', 'medium', 'long']
GEOM_JSON = '/content/ABPT/archive/qwen35_4b_anchor_geometry_profile_sweep.json'
GEOM_MD = '/content/ABPT/docs/research/qwen35_4b_anchor_geometry_profile_sweep.md'

for path in [GEOM_JSON, GEOM_MD]:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

print('MODEL_NAME =', MODEL_NAME)
print('PROFILES =', PROFILES)


In [ ]:
import subprocess
cmd = [
    'python', 'scripts/run_qwen_anchor_geometry_profile_sweep.py',
    '--model', MODEL_NAME,
    '--device', DEVICE,
    '--max_length', str(MAX_LENGTH),
    '--profiles', *PROFILES,
    '--output_json', GEOM_JSON,
    '--output_md', GEOM_MD,
]
print('RUN:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print('STDERR:')
        print(result.stderr)
    raise RuntimeError(f'geometry profile sweep failed with code {result.returncode}')


In [ ]:
payload = json.loads(Path(GEOM_JSON).read_text(encoding='utf-8'))
print('metadata:')
print(json.dumps(payload['metadata'], indent=2, ensure_ascii=False))
print()
print('curve points:')
print(json.dumps(payload['curve_points'], indent=2, ensure_ascii=False))
print()
print('best geometry profile:')
print(json.dumps(payload['best_geometry_profile'], indent=2, ensure_ascii=False))
print()
print('report head:')
print(Path(GEOM_MD).read_text(encoding='utf-8')[:9000])
